In [50]:
import pyspark
from pyspark import SparkContext

conf: pyspark.SparkConf = pyspark.SparkConf().set(
    "spark.driver.host", "localhost"
)
sc: SparkContext = SparkContext.getOrCreate()

# Set log level to reduce verbosity
sc.setLogLevel("WARN")

print("✅ Connected to Spark cluster!")
print(f"Spark Version: {sc.version}")
print(f"Master: {sc.master}")
print(f"App ID: {sc.applicationId}")


✅ Connected to Spark cluster!
Spark Version: 4.0.1
Master: local[*]
App ID: local-1763834521411


In [51]:
num_csv_path = "../data/processed/merged/num_2020.csv"
pre_csv_path = "../data/processed/merged/pre_2020.csv"
sub_csv_path = "../data/processed/merged/sub_2020.csv"
tag_csv_path = "../data/processed/merged/tag_2020.csv"


num_rdd = sc.textFile(num_csv_path)
pre_rdd = sc.textFile(pre_csv_path)
sub_rdd = sc.textFile(sub_csv_path)
tag_rdd = sc.textFile(tag_csv_path)

# print size of each RDD
print(f"Num RDD size: {num_rdd.count()}")
print(f"Pre RDD size: {pre_rdd.count()}")
print(f"Sub RDD size: {sub_rdd.count()}")
print(f"Tag RDD size: {tag_rdd.count()}")

Num RDD size: 11493263
Pre RDD size: 2746310
Sub RDD size: 24940
Tag RDD size: 298803


What are the top 5 industries with the best return over assets per quarter?

In [ ]:
# parse num line
# adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,footnote,quarter,year
# 0001564590-20-010652,AccountsPayableCurrentAndNoncurrent,us-gaap/2019,20181231,0,USD,,,607000.0,,q1,2020
# 0000753308-20-000021,LongTermDebtCurrent,us-gaap/2019,20181231,0,USD,LegalEntity=NexteraEnergyResources;,NexteraEnergyResources,602000000.0,,q1,2020
# 0001393883-20-000011,RevenueFromContractWithCustomerExcludingAssessedTax,us-gaap/2019,20181231,4,USD,BusinessSegments=Other;,,9312000.0,,q1,2020
# 0001507385-20-000034,StockRedeemedOrCalledDuringPeriodValue,us-gaap/2019,20191231,4,USD,LegalEntity=VEREITOperatingPartnershipL.P.;PartnerCapitalComponents=PreferredStock;PartnerTypeOfPartnersCapitalAccount=GeneralPartner;,,182347000.0,,q1,2020
# 0001564590-20-005569,AvailableForSaleSecuritiesDebtSecurities,us-gaap/2019,20191231,0,USD,FairValueByFairValueHierarchyLevel=FairValueInputsLevel3;FairValueByMeasurementFrequency=FairValueMeasurementsRecurring;FinancialInstrument=MortgageBackedSecurities;,,0.0,,q1,2020
# 0000075252-20-000021,IncreaseDecreaseInOtherOperatingCapitalNet,us-gaap/2019,20191231,4,USD,ConsolidatedEntities=GuarantorSubsidiaries;,,709524000.0,,q1,2020
# 0000024545-20-000005,NetCashProvidedByUsedInFinancingActivities,us-gaap/2019,20191231,4,USD,ConsolidatedEntities=NonGuarantorSubsidiaries;,,-122000000.0,,q1,2020
# 0001402057-20-000042,NetChangeInAccountsPayableInventoryFinancing,0001402057-20-000042,20191231,4,USD,ConsolidatedEntities=SubsidiaryIssuer;ConsolidationItems=ReportableLegalEntities;,,0.0,,q1,2020


class NumberEntry:
    def __init__(
        self,
        accession_number: str,
        tag: str,
        version: str,
        ddate: str,
        quarter_count: str,
        unit_of_measurement: str,
        segments: str,
        coregistrant_parent_company: str,
        value: str,
        footnote: str,
        quarter: str,
        year: str,
    ):
        self.accession_number = accession_number
        self.tag = tag
        self.version = version
        self.ddate = ddate
        self.quarter_count = quarter_count
        self.unit_of_measurement = unit_of_measurement
        self.segments = segments
        self.coregistrant_parent_company = coregistrant_parent_company
        self.value = value
        self.footnote = footnote
        self.quarter = quarter
        self.year = year


parsed_num_rdd = num_rdd.map(lambda line: line.split(",")).map(
    lambda fields: NumberEntry(
        accession_number=fields[0],
        tag=fields[1],
        version=fields[2],
        ddate=fields[3],
        quarter_count=fields[4],
        unit_of_measurement=fields[5],
        segments=fields[6],
        coregistrant_parent_company=fields[7],
        value=fields[8],
        footnote=fields[9],
        quarter=fields[10],
        year=fields[11],
    )
)

In [53]:
# pre entry
# adsh,report,line,stmt,inpth,rfile,tag,version,plabel,negating,quarter,year
# 0000002178-20-000013,2,3,BS,0,H,CashAndCashEquivalentsAtCarryingValue,us-gaap/2019,Cash and cash equivalents,0,q1,2020
# 0000002178-20-000013,2,4,BS,0,H,RestrictedCashCurrent,us-gaap/2019,Restricted cash,0,q1,2020
# 0000002178-20-000013,2,5,BS,0,H,AccountsReceivableNetCurrent,us-gaap/2019,"Accounts receivable, net of allowance for doubtful accounts of $141 and $153, respectively",0,q1,2020
# 0000002178-20-000013,2,6,BS,0,H,AccountsReceivableRelatedPartiesCurrent,us-gaap/2019,Accounts receivable  related party,0,q1,2020
class PreEntry:
    def __init__(
        self,
        adsh: str,
        report: str,
        line: str,
        stmt: str,
        inpth: str,
        rfile: str,
        tag: str,
        version: str,
        plabel: str,
        negating: str,
        quarter: str,
        year: str,
    ):
        self.adsh = adsh
        self.report = report
        self.line = line
        self.stmt = stmt
        self.inpth = inpth
        self.rfile = rfile
        self.tag = tag
        self.version = version
        self.plabel = plabel
        self.negating = negating
        self.quarter = quarter
        self.year = year

In [54]:
class SubEntry:
    def __init__(
        self,
        adsh: str,
        cik: str,
        name: str,
        sic: str,
        countryba: str,
        stprba: str,
        cityba: str,
        zipba: str,
        bas1: str,
        bas2: str,
        baph: str,
        countryma: str,
        stprma: str,
        cityma: str,
        zipma: str,
        mas1: str,
        mas2: str,
        maph: str,
        countryinc: str,
        stprinc: str,
        ein: str,
        former: str,
        changed: str,
        fiscalyearend: str,
        formersic: str,
    ):
        self.adsh = adsh
        self.cik = cik
        self.name = name
        self.sic = sic
        self.countryba = countryba
        self.stprba = stprba
        self.cityba = cityba
        self.zipba = zipba
        self.bas1 = bas1
        self.bas2 = bas2
        self.baph = baph
        self.countryma = countryma
        self.stprma = stprma
        self.cityma = cityma
        self.zipma = zipma
        self.mas1 = mas1
        self.mas2 = mas2
        self.maph = maph
        self.countryinc = countryinc
        self.stprinc = stprinc
        self.ein = ein
        self.former = former
        self.changed = changed
        self.fiscalyearend = fiscalyearend
        self.formersic = formersic

In [55]:
# tag entry
# tag,version,custom,abstract,datatype,iord,crdr,tlabel,doc,quarter,year
# OperatingLeasesRentExpenseNet,us-gaap/2018,0,0,monetary,D,D,"Operating Leases, Rent Expense, Net","Rental expense for the reporting period incurred under operating leases, including minimum and any contingent rent expense, net of related sublease income.",q1,2020
# OperatingLeaseVariableLeaseIncome,us-gaap/2018,0,0,monetary,D,C,"Operating Lease, Variable Lease Income","Amount of operating lease income from variable lease payments paid and payable to lessor, excluding amount included in measurement of lease receivable.",q1,2020
# OperatingLeaseWeightedAverageDiscountRatePercent,us-gaap/2018,0,0,percent,I,,"Operating Lease, Weighted Average Discount Rate, Percent",Weighted average discount rate for operating lease calculated at point in time.,q1,2020
# DeferredCompensationArrangementWithIndividualCompensationExpense,us-gaap/2018,0,0,monetary,D,D,"Deferred Compensation Arrangement with Individual, Compensation Expense",The compensation expense recognized during the period pertaining to the deferred compensation arrangement.,q1,2020
# DeferredCompensationEquity,us-gaap/2018,0,0,monetary,I,D,Deferred Compensation Equity,"Value of stock issued under share-based plans to employees or officers which is the unearned portion, accounted for under the fair value method.",q1,2020
# DeferredCompensationLiabilityClassifiedNoncurrent,us-gaap/2018,0,0,monetary,I,C,"Deferred Compensation Liability, Classified, Noncurrent","Aggregate carrying value as of the balance sheet date of the liabilities for all deferred compensation arrangements payable beyond one year (or the operating cycle, if longer).",q1,2020
# DeferredCompensationLiabilityCurrent,us-gaap/2018,0,0,monetary,I,C,"Deferred Compensation Liability, Current","Aggregate carrying value as of the balance sheet date of the liabilities for all deferred compensation arrangements payable within one year (or the operating cycle, if longer). Represents currently earned compensation under compensation arrangements that is not actually paid until a later date.",q1,2020
# DeferredCompensationLiabilityCurrentAndNoncurrent,us-gaap/2018,0,0,monetary,I,C,"Deferred Compensation Liability, Current and Noncurrent",Aggregate carrying value as of the balance sheet date of the liabilities for all deferred compensation arrangements. Represents currently earned compensation under compensation arrangements that is not actually paid until a later date.,q1,2020
# OriginationOfLoansToEmployeeStockOwnershipPlans,us-gaap/2018,0,0,monetary,D,C,Origination of Loans to Employee Stock Ownership Plans,"The cash outflow to finance the entity's defined contribution plan to acquire shares of the entity. The plan initially holds the shares in a suspense account, which is collateral for the loan. As the plan makes payment on the debt, the shares are released from the suspense account and become available to be allocated to participant accounts.",q1,2020


class TagEntry:
    def __init__(
        self,
        tag: str,
        version: str,
        custom: str,
        abstract: str,
        datatype: str,
        iord: str,
        crdr: str,
        tlabel: str,
        doc: str,
        quarter: str,
        year: str,
    ):
        self.tag = tag
        self.version = version
        self.custom = custom
        self.abstract = abstract
        self.datatype = datatype
        self.iord = iord
        self.crdr = crdr
        self.tlabel = tlabel
        self.doc = doc
        self.quarter = quarter
        self.year = year

In [ ]:
parsed_num_rdd = num_rdd.map(lambda line: line.split(",")).map(
    lambda fields: NumberEntry(
        accession_number=fields[0],
        tag=fields[1],
        version=fields[2],
        ddate=fields[3],
        quarter_count=fields[4],
        unit_of_measurement=fields[5],
        segments=fields[6],
        coregistrant_parent_company=fields[7],
        value=fields[8],
        footnote=fields[9],
        quarter=fields[10],
        year=fields[11],
    )
)

parsed_sub_rdd = sub_rdd.map(lambda line: line.split(",")).map(
    lambda fields: SubEntry(
        adsh=fields[0],
        cik=fields[1],
        name=fields[2],
        sic=fields[3],
        countryba=fields[4],
        stprba=fields[5],
        cityba=fields[6],
        zipba=fields[7],
        bas1=fields[8],
        bas2=fields[9],
        baph=fields[10],
        countryma=fields[11],
        stprma=fields[12],
        cityma=fields[13],
        zipma=fields[14],
        mas1=fields[15],
        mas2=fields[16],
        maph=fields[17],
        countryinc=fields[18],
        stprinc=fields[19],
        ein=fields[20],
        former=fields[21],
        changed=fields[22],
        fiscalyearend=fields[23],
        formersic=fields[24],
    )
)

parsed_pre_rdd = pre_rdd.map(lambda line: line.split(",")).map(
    lambda fields: PreEntry(
        adsh=fields[0],
        report=fields[1],
        line=fields[2],
        stmt=fields[3],
        inpth=fields[4],
        rfile=fields[5],
        tag=fields[6],
        version=fields[7],
        plabel=fields[8],
        negating=fields[9],
        quarter=fields[10],
        year=fields[11],
    )
)

parsed_tag_rdd = tag_rdd.map(lambda line: line.split(",")).map(
    lambda fields: TagEntry(
        tag=fields[0],
        version=fields[1],
        custom=fields[2],
        abstract=fields[3],
        datatype=fields[4],
        iord=fields[5],
        crdr=fields[6],
        tlabel=fields[7],
        doc=fields[8],
        quarter=fields[9],
        year=fields[10],
    )
)


for entry in parsed_num_rdd.take(5):
    print(
        f"ADSH: {entry.accession_number}, Tag: {entry.tag}, Date: {entry.ddate}, Value: {entry.value}"
    )
for entry in parsed_sub_rdd.take(5):
    print(f"ADSH: {entry.adsh}, CIK: {entry.cik}, Name: {entry.name}")

for entry in parsed_pre_rdd.take(5):
    print(f"ADSH: {entry.adsh}, Tag: {entry.tag}, Plabel: {entry.plabel}")

for entry in parsed_tag_rdd.take(5):
    print(f"Tag: {entry.tag}, TLabel: {entry.tlabel}, Doc: {entry.doc}")
# --- IGNORE ---

ADSH: adsh, Tag: tag, Date: ddate, Value: value
ADSH: 0001564590-20-010652, Tag: AccountsPayableCurrentAndNoncurrent, Date: 20181231, Value: 607000.0
ADSH: 0000753308-20-000021, Tag: LongTermDebtCurrent, Date: 20181231, Value: 602000000.0
ADSH: 0001393883-20-000011, Tag: RevenueFromContractWithCustomerExcludingAssessedTax, Date: 20181231, Value: 9312000.0
ADSH: 0001507385-20-000034, Tag: StockRedeemedOrCalledDuringPeriodValue, Date: 20191231, Value: 182347000.0
ADSH: adsh, CIK: cik, Name: name
ADSH: 0000002178-20-000013, CIK: 2178, Name: "ADAMS RESOURCES & ENERGY
ADSH: 0000002488-20-000008, CIK: 2488, Name: ADVANCED MICRO DEVICES INC
ADSH: 0000002969-20-000010, CIK: 2969, Name: AIR PRODUCTS & CHEMICALS INC /DE/
ADSH: 0000003499-20-000005, CIK: 3499, Name: ALEXANDERS INC
ADSH: adsh, Tag: tag, Plabel: plabel
ADSH: 0000002178-20-000013, Tag: CashAndCashEquivalentsAtCarryingValue, Plabel: Cash and cash equivalents
ADSH: 0000002178-20-000013, Tag: RestrictedCashCurrent, Plabel: Restricted c

In [57]:
# define the tags we are interested in for income and assets
net_income_desired_tag_list = [
    "NetIncomeLoss",
    "ProfitLoss",
    "NetIncomeLossAvailableToCommonStockholdersBasic",
    "NetIncomeLossAvailableToCommonStockholdersDiluted",
]
asset_tags = [
    "Assets",
    "CashAndCashEquivalentsAtCarryingValue",
    "AccountsReceivableNetCurrent",
    "InventoryNet",
    "PropertyPlantAndEquipmentNet",
    "Goodwill",
    "IntangibleAssetsNetExcludingGoodwill",
    "LongTermInvestments",
]


In [ ]:
res = (
    parsed_num_rdd.filter(lambda x: x.tag in net_income_desired_tag_list)
    .filter(lambda x: x.unit_of_measurement == "USD")
    .filter(lambda x: x.coregistrant_parent_company == "")
    .filter(lambda x: x.segments == "")
    .map(lambda x: [x.accession_number, x.tag, x.value, x.quarter])
    .take(5)
)

res

[['0001564590-20-010998', 'ProfitLoss', '-41662000.0', 'q1'],
 ['0001239819-20-000032', 'NetIncomeLoss', '11004241.0', 'q1'],
 ['0001572758-20-000018', 'NetIncomeLoss', '3209000.0', 'q1'],
 ['0001637207-20-000008', 'ProfitLoss', '135413000.0', 'q1'],
 ['0000766829-20-000055', 'ProfitLoss', '38767000.0', 'q1']]